# Day 7 - Initial Dataset Inspection

This notebook performs read-only inspection of the untouched archive. It inventories files, images, YOLO labels, metadata-like files, sequence naming, duplicates, class coverage, and obvious inconsistencies. It does not modify the raw archive.

Place the three downloaded dataset folders in `raw_data_archive/` before running the code cells.

In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import json
import re

ROOT = Path.cwd()
ARCHIVE = ROOT / 'raw_data_archive'
REPORTS = ROOT / 'inspection_outputs'
REPORTS.mkdir(exist_ok=True)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
LABEL_EXTENSIONS = {'.txt'}
print('Working directory:', ROOT)
print('Archive exists:', ARCHIVE.exists())
print('Archive entries:', sorted(p.name for p in ARCHIVE.iterdir()) if ARCHIVE.exists() else [])

Working directory: /Users/dakarapuhemamdharnath/Desktop/hemamdhar/day7_hemamdhar_initialinspection
Archive exists: True
Archive entries: ['README.md', 'SOURCE_LINK.txt']


In [3]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def source_video_id(filename):
    stem = Path(filename).stem
    match = re.match(r'(.+?_mp4)', stem, flags=re.IGNORECASE)
    return match.group(1) if match else None

def frame_marker(filename):
    stem = Path(filename).stem
    markers = re.findall(r'(?:-|_)\d+(?:$|_)', stem)
    return markers[-1].strip('-_') if markers else None

def parse_yolo_label(path, class_count=None):
    errors = []
    rows = []
    try:
        lines = path.read_text(encoding='utf-8').splitlines()
    except Exception as exc:
        return [], [f'read_error:{type(exc).__name__}']
    for line_number, line in enumerate(lines, 1):
        parts = line.split()
        if not parts:
            continue
        if len(parts) != 5:
            errors.append(f'line_{line_number}:expected_5_fields')
            continue
        try:
            class_id = int(parts[0])
            values = [float(value) for value in parts[1:]]
        except ValueError:
            errors.append(f'line_{line_number}:non_numeric')
            continue
        if class_id < 0 or (class_count is not None and class_id >= class_count):
            errors.append(f'line_{line_number}:invalid_class_id_{class_id}')
        if any(value < 0 or value > 1 for value in values):
            errors.append(f'line_{line_number}:coordinate_out_of_range')
        if values[2] <= 0 or values[3] <= 0:
            errors.append(f'line_{line_number}:non_positive_box_size')
        rows.append((class_id, *values))
    return rows, errors

def read_class_names(dataset_root):
    candidates = list(dataset_root.rglob('data.yaml')) + list(dataset_root.rglob('data.yml'))
    for config in candidates:
        names = {}
        for raw_line in config.read_text(encoding='utf-8', errors='replace').splitlines():
            match = re.match(r'\s*(?:names\s*:\s*)?(\d+)\s*:\s*["]?(.+?)["]?\s*$', raw_line)
            if match:
                names[int(match.group(1))] = match.group(2).strip(" \"'")
        if names:
            return names, str(config)
    return {}, None

In [4]:
records = []
if ARCHIVE.exists():
    for dataset_root in sorted(p for p in ARCHIVE.iterdir() if p.is_dir()):
        names, config_path = read_class_names(dataset_root)
        all_files = [p for p in dataset_root.rglob('*') if p.is_file()]
        images = [p for p in all_files if p.suffix.lower() in IMAGE_EXTENSIONS]
        labels = [p for p in all_files if p.suffix.lower() in LABEL_EXTENSIONS and p.parent.name == 'labels']
        videos = [p for p in all_files if p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}]
        records.append({
            'dataset': dataset_root.name,
            'files': len(all_files),
            'images': len(images),
            'labels': len(labels),
            'videos': len(videos),
            'classes': len(names),
            'class_names': names,
            'data_config': config_path,
        })
print(json.dumps(records, indent=2))
(REPORTS / 'inventory.json').write_text(json.dumps(records, indent=2), encoding='utf-8')

[]


2

In [5]:
label_records = []
class_counts = Counter()
error_counts = Counter()
sequence_counts = Counter()
image_stems = defaultdict(list)
for dataset_root in sorted(p for p in ARCHIVE.iterdir() if p.is_dir()) if ARCHIVE.exists() else []:
    names, _ = read_class_names(dataset_root)
    label_paths = [p for p in dataset_root.rglob('labels/*.txt')]
    for label_path in label_paths:
        rows, errors = parse_yolo_label(label_path, len(names) if names else None)
        image_stem = label_path.stem
        image_candidates = [p for p in label_path.parents[1].glob(f'{image_stem}.*') if p.suffix.lower() in IMAGE_EXTENSIONS]
        for row in rows:
            class_counts[(dataset_root.name, row[0])] += 1
        for error in errors:
            error_counts[(dataset_root.name, error.split(':', 1)[-1])] += 1
        sequence_id = source_video_id(image_stem)
        if sequence_id:
            sequence_counts[(dataset_root.name, sequence_id)] += 1
        image_stems[(dataset_root.name, image_stem)].append(str(image_candidates[0]) if image_candidates else None)
        label_records.append({
            'dataset': dataset_root.name,
            'label': str(label_path),
            'objects': len(rows),
            'errors': errors,
            'source_video_id': sequence_id,
            'frame_marker': frame_marker(image_stem),
            'matching_image': str(image_candidates[0]) if image_candidates else None,
        })
print('Class counts:', dict(class_counts))
print('Label errors:', dict(error_counts))
print('Sequence groups:', len(sequence_counts))
(REPORTS / 'label_records.json').write_text(json.dumps(label_records, indent=2), encoding='utf-8')
(REPORTS / 'class_counts.json').write_text(json.dumps({f'{dataset}|{class_id}': count for (dataset, class_id), count in class_counts.items()}, indent=2), encoding='utf-8')

Class counts: {}
Label errors: {}
Sequence groups: 0


2

In [6]:
# Optional image inspection. Pillow is not required for the inventory cells above.
try:
    from PIL import Image
except ImportError:
    Image = None

image_stats = []
if Image is not None and ARCHIVE.exists():
    for image_path in ARCHIVE.rglob('*'):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                with Image.open(image_path) as image:
                    image_stats.append({'path': str(image_path), 'format': image.format, 'width': image.width, 'height': image.height, 'mode': image.mode})
            except Exception as exc:
                image_stats.append({'path': str(image_path), 'error': type(exc).__name__})
print('Image records inspected:', len(image_stats))
print('Pillow available:', Image is not None)
(REPORTS / 'image_stats.json').write_text(json.dumps(image_stats, indent=2), encoding='utf-8')

Image records inspected: 0
Pillow available: True


2

In [7]:
# Read-only duplicate screening by file size and SHA-256.
hash_groups = defaultdict(list)
if ARCHIVE.exists():
    for path in ARCHIVE.rglob('*'):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                hash_groups[(path.stat().st_size, sha256(path))].append(str(path))
            except OSError:
                pass
duplicates = [paths for paths in hash_groups.values() if len(paths) > 1]
print('Exact duplicate groups:', len(duplicates))
(REPORTS / 'exact_duplicates.json').write_text(json.dumps(duplicates, indent=2), encoding='utf-8')

Exact duplicate groups: 0


2

## Interpretation Notes

Review `inspection_outputs/` after execution. Treat missing labels, invalid coordinates, unmatched image-label pairs, empty annotations, duplicate hashes, inconsistent dimensions, and source-sequence overlap as findings to document. Do not repair them in this notebook.